# **Building a model to classify *blood cell images***

https://www.kaggle.com/datasets/paultimothymooney/blood-cells

- **Context**: The diagnosis of blood-based diseases often involves identifying and characterizing patient blood samples. Automated methods to detect and classify blood cell subtypes have important medical applications.

- **Content**: This dataset contains 12,500 augmented images of blood cells (JPEG) with accompanying cell type labels (CSV). There are approximately 3,000 images for each of 4 different cell types grouped into 4 different folders (according to cell type). The cell types are Eosinophil, Lymphocyte, Monocyte, and Neutrophil. This dataset is accompanied by an additional dataset containing the original 410 images (pre-augmentation) as well as two additional subtype labels (WBC vs WBC) and also bounding boxes for each cell in each of these 410 images (JPEG + XML metadata). More specifically, the folder 'dataset-master' contains 410 images of blood cells with subtype labels and bounding boxes (JPEG + XML), while the folder 'dataset2-master' contains 2,500 augmented images as well as 4 additional subtype labels (JPEG + CSV). There are approximately 3,000 augmented images for each class of the 4 classes as compared to 88, 33, 21, and 207 images of each in folder 'dataset-master'.

## **1. Data**

In [116]:
import os
from PIL import Image
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from google.colab import drive
from torchmetrics.classification import Accuracy, Precision, Recall, F1Score

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [117]:
class BloodCellDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = ['EOSINOPHIL', 'LYMPHOCYTE', 'MONOCYTE', 'NEUTROPHIL']
        self.image_paths = []
        self.labels = []

        for idx, class_name in enumerate(self.classes):
            class_dir = os.path.join(root_dir, class_name)

            if not os.path.isdir(class_dir):
                continue

            for img_name in os.listdir(class_dir):
                if img_name.endswith(('.png', '.jpg', '.jpeg')):
                    self.image_paths.append(os.path.join(class_dir, img_name))
                    self.labels.append(idx)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [118]:
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomRotation(360),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomAdjustSharpness(sharpness_factor=0.2, p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [119]:
test_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [120]:
full_train_dataset = BloodCellDataset(
    root_dir='/content/drive/MyDrive/data/blood_cells/TRAIN',
    transform=train_transform
)
full_val_dataset = BloodCellDataset(
    root_dir='/content/drive/MyDrive/data/blood_cells/TRAIN',
    transform=test_transform
)

In [121]:
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size

In [122]:
generator = torch.Generator().manual_seed(42)
train_subset, _ = random_split(full_train_dataset, [train_size, val_size], generator=generator)
_, val_subset = random_split(full_val_dataset, [train_size, val_size], generator=generator)

In [123]:
train_dataloader = DataLoader(
    train_subset, batch_size=32, shuffle=True, num_workers=os.cpu_count()
)
val_dataloader = DataLoader(
    val_subset, batch_size=32, shuffle=False, num_workers=os.cpu_count()
)

In [124]:
test_dataset = BloodCellDataset(root_dir='/content/drive/MyDrive/data/blood_cells/TEST', transform=test_transform)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=os.cpu_count())

---

## **2. Model**

#

In [125]:
class BloodCellCNN(nn.Module):
    def __init__(self, input_shape=3, hidden_units=32, num_classes=4):
        super().__init__()

        self.conv_layer_1 = nn.Sequential(
            nn.Conv2d(in_channels=input_shape, out_channels=hidden_units, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden_units),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden_units),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.conv_layer_2 = nn.Sequential(
            nn.Conv2d(in_channels=hidden_units, out_channels=hidden_units * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden_units * 2),
            nn.ReLU(),
            nn.Conv2d(in_channels=hidden_units * 2, out_channels=hidden_units * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden_units * 2),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.classification_layer = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(in_features=(hidden_units * 2) * 4 * 4, out_features=256),
            nn.ReLU(),
            nn.Dropout(p=0.4),
            nn.Linear(in_features=256, out_features=num_classes)
        )

    def forward(self, x):
        x = self.conv_layer_1(x)
        x = self.conv_layer_2(x)
        x = self.classification_layer(x)
        return x

In [126]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [127]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)
model = BloodCellCNN(input_shape=3, hidden_units=64, num_classes=4).to(device)

In [128]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
accuracy = Accuracy(task="multiclass", num_classes=4).to(device)
precision = Precision(task="multiclass", num_classes=4, average='macro').to(device)
recall = Recall(task="multiclass", num_classes=4, average='macro').to(device)
f1 = F1Score(task="multiclass", num_classes=4, average='macro').to(device)

In [129]:
def train_model(model, train_data, val_data, epochs):
    best_val_loss = float('inf')

    for epoch in range(epochs):
        model.train()
        train_loss, train_acc, train_f1, train_rec = 0, 0, 0, 0

        for X_batch, y_batch in train_data:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            y_preds = model(X_batch)
            loss = criterion(y_preds, y_batch)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_acc += accuracy(y_preds, y_batch).item()
            train_f1 += f1(y_preds, y_batch).item()
            train_rec += recall(y_preds, y_batch).item()

        train_loss /= len(train_data)
        train_acc /= len(train_data)
        train_f1 /= len(train_data)
        train_rec /= len(train_data)

        model.eval()
        val_loss, val_acc, val_f1, val_rec = 0, 0, 0, 0

        with torch.inference_mode():
            for X_batch, y_batch in val_data:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                y_val_preds = model(X_batch)

                val_loss += criterion(y_val_preds, y_batch).item()
                val_acc += accuracy(y_val_preds, y_batch).item()
                val_f1 += f1(y_val_preds, y_batch).item()
                val_rec += recall(y_val_preds, y_batch).item()

        val_loss /= len(val_data)
        val_acc /= len(val_data)
        val_f1 /= len(val_data)
        val_rec /= len(val_data)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'best_weights.pth')
            print(f"*** Zapisano model (Val Loss: {val_loss:.4f}) ***")

        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Loss: {train_loss:.4f}, Acc: {train_acc*100:.2f}%, F1: {train_f1:.4f} | "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc*100:.2f}%, Val F1: {val_f1:.4f}, Val Recall: {val_rec:.4f}")
        print("-" * 70)

In [130]:
def test_model(model, test_data):
    test_loss, test_acc, test_f1, test_rec, test_prec = 0, 0, 0, 0, 0

    model.eval()
    with torch.inference_mode():
        for X_batch, y_batch in test_data:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_test_preds = model(X_batch)

            test_loss += criterion(y_test_preds, y_batch).item()
            test_acc += accuracy(y_test_preds, y_batch).item()
            test_f1 += f1(y_test_preds, y_batch).item()
            test_rec += recall(y_test_preds, y_batch).item()
            test_prec += precision(y_test_preds, y_batch).item()

    test_loss /= len(test_data)
    test_acc /= len(test_data)
    test_f1 /= len(test_data)
    test_rec /= len(test_data)
    test_prec /= len(test_data)

    print("\n" + "="*40)
    print("Final results on test dataset: \n")
    print("="*40)
    print(f"Loss:      {test_loss:.4f}")
    print(f"Accuracy:  {test_acc*100:.2f}%")
    print(f"Precision: {test_prec:.4f}")
    print(f"Recall:    {test_rec:.4f}")
    print(f"F1-Score:  {test_f1:.4f}")
    print("="*40)

In [ ]:
train_model(model=model, train_data=train_dataloader, val_data=val_dataloader, epochs=15)
model.load_state_dict(torch.load('best_weights.pth', weights_only=True))
test_model(model=model, test_data=test_dataloader)